# 🖥️ 05 – Dashboard Streamlit: AI Pharma Monitoring
**Proyek:** AI-Based Pharmaceutical Data Selection & Monitoring  
**Tim:** PJK-GM016 | Pijak × IBM SkillsBuild  
**Minggu:** 4–5 – Dashboard Interaktif & Integrasi Sistem

---
### Struktur Tab Dashboard
| Tab | Konten |
|-----|--------|
| 🏠 Overview | KPI cards, pie chart, defect rate per produk |
| 📈 Monitoring | Tren yield %, moving average, waste tracking |
| 🤖 Prediksi | Form input → prediksi Normal/Defect + gauge chart |
| 📦 Clustering | PCA scatter, profil klaster, heatmap |
| 💡 Insight | Feature importance, rekomendasi, korelasi |

> ⚠️ **Prasyarat:** Jalankan 03_modeling.ipynb & 04_clustering.ipynb lebih dulu.


## ⚙️ 1. Cek Dependensi & File

In [5]:
import os, json, pandas as pd, numpy as np

print("=" * 55)
print("  CEK DEPENDENSI DASHBOARD")
print("=" * 55)

model_files = [
    'models/random_forest_model.pkl', 'models/isolation_forest_model.pkl',
    'models/scaler.pkl', 'models/feature_cols.txt',
    'models/kmeans_model.pkl', 'models/scaler_clustering.pkl',
    'models/pca_model.pkl', 'models/cluster_features.txt',
    'models/cluster_labels_map.json',
]
data_files = [
    'rekap_produksi_clean.csv',
    'rekap_produksi_with_predictions.csv',
    'rekap_produksi_clustered.csv',
]

print("\n📁 Model Files:")
for f in model_files:
    status = "✅" if os.path.exists(f) else "❌ BELUM ADA — jalankan nb 03 & 04"
    print(f"   {status}  {f}")

print("\n📊 Data Files:")
for f in data_files:
    status = "✅" if os.path.exists(f) else "❌ BELUM ADA"
    print(f"   {status}  {f}")

print("\n📦 Libraries:")
for lib in ['streamlit','plotly','sklearn','joblib','pandas']:
    try:
        m = __import__(lib)
        v = getattr(m,'__version__','ok')
        print(f"   ✅ {lib}: {v}")
    except ImportError:
        print(f"   ❌ {lib}: pip install {lib}")

  CEK DEPENDENSI DASHBOARD

📁 Model Files:
   ✅  models/random_forest_model.pkl
   ✅  models/isolation_forest_model.pkl
   ✅  models/scaler.pkl
   ✅  models/feature_cols.txt
   ✅  models/kmeans_model.pkl
   ✅  models/scaler_clustering.pkl
   ✅  models/pca_model.pkl
   ✅  models/cluster_features.txt
   ✅  models/cluster_labels_map.json

📊 Data Files:
   ✅  rekap_produksi_clean.csv
   ✅  rekap_produksi_with_predictions.csv
   ✅  rekap_produksi_clustered.csv

📦 Libraries:
   ❌ streamlit: pip install streamlit
   ❌ plotly: pip install plotly
   ✅ sklearn: 1.7.1
   ✅ joblib: 1.5.3
   ✅ pandas: 3.0.2


## 🏗️ 2. Generate File `app.py`

In [6]:
# Cell ini menulis app.py ke disk.
# Setelah selesai: buka terminal -> streamlit run app.py

import os

dashboard_lines = [
    "import streamlit as st",
    "import pandas as pd",
    "import numpy as np",
    "import plotly.express as px",
    "import plotly.graph_objects as go",
    "import joblib, json, os, warnings",
    "warnings.filterwarnings(\"ignore\")",
    "",
    "st.set_page_config(",
    "    page_title=\"AI Pharma Monitoring | PJK-GM016\",",
    "    page_icon=\"💊\",",
    "    layout=\"wide\",",
    "    initial_sidebar_state=\"expanded\"",
    ")",
    "",
    "# -- Custom CSS ------------------------------------------------",
    "card_css = \"\"\"",
    "<style>",
    ".kpi-blue   { background:linear-gradient(135deg,#667eea,#764ba2); padding:18px;",
    "              border-radius:12px; color:white; text-align:center; }",
    ".kpi-red    { background:linear-gradient(135deg,#f093fb,#e74c3c); padding:18px;",
    "              border-radius:12px; color:white; text-align:center; }",
    ".kpi-green  { background:linear-gradient(135deg,#43e97b,#38f9d7); padding:18px;",
    "              border-radius:12px; color:white; text-align:center; }",
    ".kpi-orange { background:linear-gradient(135deg,#f7971e,#ffd200); padding:18px;",
    "              border-radius:12px; color:white; text-align:center; }",
    ".kpi-value  { font-size:2rem; font-weight:bold; }",
    ".kpi-label  { font-size:0.9rem; opacity:0.9; }",
    "</style>",
    "\"\"\"",
    "st.markdown(card_css, unsafe_allow_html=True)",
    "",
    "# -- Load Data -------------------------------------------------",
    "@st.cache_data",
    "def load_data():",
    "    for f in [\"rekap_produksi_with_predictions.csv\",\"rekap_produksi_clustered.csv\",\"rekap_produksi_clean.csv\"]:",
    "        if os.path.exists(f):",
    "            return pd.read_csv(f, low_memory=False)",
    "    st.error(\"Data tidak ditemukan.\")",
    "    st.stop()",
    "",
    "@st.cache_resource",
    "def load_models():",
    "    m = {}",
    "    try:",
    "        m[\"rf\"]     = joblib.load(\"models/random_forest_model.pkl\")",
    "        m[\"iso\"]    = joblib.load(\"models/isolation_forest_model.pkl\")",
    "        m[\"scaler\"] = joblib.load(\"models/scaler.pkl\")",
    "        m[\"feats\"]  = open(\"models/feature_cols.txt\").read().splitlines()",
    "        m[\"kmeans\"] = joblib.load(\"models/kmeans_model.pkl\")",
    "        m[\"sc_cl\"]  = joblib.load(\"models/scaler_clustering.pkl\")",
    "        m[\"pca\"]    = joblib.load(\"models/pca_model.pkl\")",
    "        m[\"cl_feats\"] = open(\"models/cluster_features.txt\").read().splitlines()",
    "        m[\"cl_map\"] = json.load(open(\"models/cluster_labels_map.json\"))",
    "    except Exception as e:",
    "        st.warning(f\"Model load warning: {e}\")",
    "    return m",
    "",
    "df  = load_data()",
    "mdl = load_models()",
    "",
    "# -- Sidebar ---------------------------------------------------",
    "with st.sidebar:",
    "    st.title(\"💊 AI Pharma Monitor\")",
    "    st.markdown(\"**PJK-GM016** | Pijak × IBM SkillsBuild\")",
    "    st.divider()",
    "    products = [\"Semua\"] + sorted(df[\"Material_Description\"].dropna().unique().tolist())",
    "    sel_prod = st.selectbox(\"🔽 Produk\", products)",
    "    if sel_prod != \"Semua\":",
    "        df = df[df[\"Material_Description\"] == sel_prod]",
    "    if \"Bulan_Produksi\" in df.columns:",
    "        months = sorted(df[\"Bulan_Produksi\"].dropna().unique().astype(int).tolist())",
    "        sel_m  = st.multiselect(\"📅 Bulan\", months, default=months,",
    "                                 format_func=lambda x: f\"Bulan {x}\")",
    "        if sel_m:",
    "            df = df[df[\"Bulan_Produksi\"].isin(sel_m)]",
    "",
    "# -- Header ----------------------------------------------------",
    "st.title(\"🏭 AI-Based Pharmaceutical Production Monitoring\")",
    "st.markdown(\"> *Sistem monitoring berbasis AI untuk deteksi defect & analisis kualitas produksi*\")",
    "st.divider()",
    "",
    "tab1, tab2, tab3, tab4, tab5 = st.tabs(",
    "    [\"🏠 Overview\", \"📈 Monitoring\", \"🤖 Prediksi\", \"📦 Clustering\", \"💡 Insight\"]",
    ")",
    "",
    "# ==== TAB 1 OVERVIEW ==========================================",
    "with tab1:",
    "    st.subheader(\"📊 Ringkasan Eksekutif\")",
    "    total  = len(df)",
    "    n_def  = int(df[\"Defect_Overall\"].sum()) if \"Defect_Overall\" in df.columns else 0",
    "    dr     = n_def/total*100 if total else 0",
    "    avg_ct = df[\"Cetak_Pct_Teoritis\"].mean()*100 if \"Cetak_Pct_Teoritis\" in df.columns else 0",
    "    avg_km = df[\"Kemas_Pct_Teoritis\"].mean()*100 if \"Kemas_Pct_Teoritis\" in df.columns else 0",
    "",
    "    c1,c2,c3,c4 = st.columns(4)",
    "    with c1:",
    "        st.markdown(f'<div class=\"kpi-blue\"><div class=\"kpi-value\">{total:,}</div><div class=\"kpi-label\">Total Batch</div></div>', unsafe_allow_html=True)",
    "    with c2:",
    "        st.markdown(f'<div class=\"kpi-red\"><div class=\"kpi-value\">{dr:.1f}%</div><div class=\"kpi-label\">Defect Rate</div></div>', unsafe_allow_html=True)",
    "    with c3:",
    "        st.markdown(f'<div class=\"kpi-green\"><div class=\"kpi-value\">{avg_ct:.1f}%</div><div class=\"kpi-label\">Avg Yield Cetak</div></div>', unsafe_allow_html=True)",
    "    with c4:",
    "        st.markdown(f'<div class=\"kpi-orange\"><div class=\"kpi-value\">{avg_km:.1f}%</div><div class=\"kpi-label\">Avg Yield Kemas</div></div>', unsafe_allow_html=True)",
    "",
    "    st.markdown(\"---\")",
    "    cL,cR = st.columns(2)",
    "    with cL:",
    "        fig_pie = px.pie(",
    "            names=[\"Normal\",\"Defect\"],",
    "            values=[total-n_def, n_def],",
    "            color=[\"Normal\",\"Defect\"],",
    "            color_discrete_map={\"Normal\":\"#2ecc71\",\"Defect\":\"#e74c3c\"},",
    "            title=\"Normal vs Defect\", hole=0.4",
    "        )",
    "        fig_pie.update_traces(textinfo=\"percent+label+value\")",
    "        st.plotly_chart(fig_pie, use_container_width=True)",
    "    with cR:",
    "        mat_def = (df.groupby(\"Material_Description\")[\"Defect_Overall\"]",
    "                   .agg([\"sum\",\"count\"]).rename(columns={\"sum\":\"Defect\",\"count\":\"Total\"})",
    "                   .assign(Rate=lambda x: x[\"Defect\"]/x[\"Total\"]*100)",
    "                   .sort_values(\"Rate\", ascending=True))",
    "        fig_b = px.bar(",
    "            mat_def.reset_index(), x=\"Rate\", y=\"Material_Description\",",
    "            orientation=\"h\", title=\"Defect Rate per Produk (%)\",",
    "            color=\"Rate\", color_continuous_scale=[\"#2ecc71\",\"#f39c12\",\"#e74c3c\"]",
    "        )",
    "        fig_b.update_layout(coloraxis_showscale=False)",
    "        st.plotly_chart(fig_b, use_container_width=True)",
    "",
    "# ==== TAB 2 MONITORING ========================================",
    "with tab2:",
    "    st.subheader(\"📈 Monitoring Tren Yield\")",
    "    cA,cB = st.columns(2)",
    "    with cA:",
    "        thr = st.slider(\"Threshold Defect (%)\", 80, 98, 90, 1) / 100",
    "    with cB:",
    "        n_ma = st.slider(\"Moving Average (batch)\", 1, 20, 5)",
    "",
    "    for col_y, lbl in [(\"Cetak_Pct_Teoritis\",\"Cetak\"),(\"Kemas_Pct_Teoritis\",\"Kemas\")]:",
    "        if col_y not in df.columns:",
    "            continue",
    "        s    = df[col_y].dropna().reset_index(drop=True)",
    "        s_ma = s.rolling(n_ma, min_periods=1).mean()",
    "        fig_t = go.Figure()",
    "        fig_t.add_trace(go.Scatter(y=s, mode=\"lines\", name=\"% Teoritis\", line=dict(color=\"#3498db\", width=1), opacity=0.6))",
    "        fig_t.add_trace(go.Scatter(y=s_ma, mode=\"lines\", name=f\"MA({n_ma})\", line=dict(color=\"#e67e22\", width=2)))",
    "        fig_t.add_hline(y=thr, line_dash=\"dash\", line_color=\"red\", annotation_text=f\"Threshold={thr:.0%}\")",
    "        di = s[s < thr].index.tolist()",
    "        if di:",
    "            fig_t.add_trace(go.Scatter(x=di, y=s[di], mode=\"markers\", name=\"Defect\", marker=dict(color=\"red\", size=6, symbol=\"x\")))",
    "        fig_t.update_layout(title=f\"Tren Yield {lbl}\", xaxis_title=\"Batch\", yaxis_title=\"% Teoritis\", height=340)",
    "        st.plotly_chart(fig_t, use_container_width=True)",
    "",
    "    if \"Total_Waste_Kg\" in df.columns:",
    "        wv = df[\"Total_Waste_Kg\"].dropna().reset_index(drop=True)",
    "        fig_w = px.bar(",
    "            x=wv.index, y=wv.values, title=\"Waste per Batch (Kg)\",",
    "            color=wv.values, color_continuous_scale=\"YlOrRd\",",
    "            labels={\"x\":\"Batch\",\"y\":\"Waste (Kg)\"}",
    "        )",
    "        fig_w.update_layout(height=280, coloraxis_showscale=False)",
    "        st.plotly_chart(fig_w, use_container_width=True)",
    "",
    "# ==== TAB 3 PREDIKSI ==========================================",
    "with tab3:",
    "    st.subheader(\"🤖 Prediksi Kualitas Batch Baru\")",
    "    st.info(\"Masukkan parameter batch -> model memprediksi Normal atau Defect.\")",
    "    if \"rf\" not in mdl:",
    "        st.error(\"Model belum dimuat. Jalankan notebook 03 terlebih dahulu.\")",
    "    else:",
    "        feats = mdl[\"feats\"]",
    "        defaults = {",
    "            \"GB_Yield_Total\":700, \"GK_Yield_Total\":650, \"Rasio_GK_GB\":0.93,",
    "            \"GB_Kadar_Air_Mean\":4.5, \"Cetak_Yield_Kg\":780,",
    "            \"Cetak_Pct_Teoritis\":0.975, \"Kemas_Pct_Teoritis\":0.97,",
    "            \"Total_Waste_Kg\":0, \"Cetak_Durasi_Hari\":1,",
    "            \"Kemas_Durasi_Hari\":1, \"Bulan_Produksi\":6",
    "        }",
    "        num_feats = [f for f in feats if not f.endswith(\"_enc\")]",
    "        cols3 = st.columns(3)",
    "        inp = {}",
    "        for i, feat in enumerate(num_feats):",
    "            with cols3[i % 3]:",
    "                inp[feat] = st.number_input(feat.replace(\"_\", \" \"), value=float(defaults.get(feat, 0)), format=\"%.3f\")",
    "        for feat in [f for f in feats if f.endswith(\"_enc\")]:",
    "            inp[feat] = 0.0",
    "",
    "        if st.button(\"🔍 Analisis Batch\", type=\"primary\", use_container_width=True):",
    "            X_in = np.array([[inp.get(f, 0) for f in feats]])",
    "            X_sc = mdl[\"scaler\"].transform(X_in)",
    "            pred   = mdl[\"rf\"].predict(X_sc)[0]",
    "            proba  = mdl[\"rf\"].predict_proba(X_sc)[0][1]",
    "            pred_i = (mdl[\"iso\"].predict(X_sc)[0] == -1)",
    "",
    "            r1, r2, r3 = st.columns(3)",
    "            status_rf = \"🔴 DEFECT\" if pred == 1 else \"🟢 NORMAL\"",
    "            status_iso = \"⚠️ ANOMALI\" if pred_i else \"✅ Normal\"",
    "",
    "            with r1:",
    "                (st.error if pred == 1 else st.success)(f\"{status_rf} — Random Forest\")",
    "            with r2:",
    "                st.metric(\"Probabilitas Defect\", f\"{proba*100:.1f}%\", delta=f\"{(proba-0.5)*100:+.1f}% vs threshold\")",
    "            with r3:",
    "                (st.warning if pred_i else st.info)(f\"{status_iso} — Isolation Forest\")",
    "",
    "            fig_g = go.Figure(go.Indicator(",
    "                mode=\"gauge+number\",",
    "                value=proba*100,",
    "                number={\"suffix\":\"%\"},",
    "                title={\"text\":\"Probabilitas Defect\"},",
    "                gauge={",
    "                    \"axis\":{\"range\":[0,100]},",
    "                    \"bar\":{\"color\":\"#e74c3c\" if proba > 0.5 else \"#2ecc71\"},",
    "                    \"steps\":[",
    "                        {\"range\":[0,30],\"color\":\"#d5f5e3\"},",
    "                        {\"range\":[30,70],\"color\":\"#fdebd0\"},",
    "                        {\"range\":[70,100],\"color\":\"#fadbd8\"}",
    "                    ],",
    "                    \"threshold\":{\"line\":{\"color\":\"black\",\"width\":3},\"value\":50}",
    "                }",
    "            ))",
    "            fig_g.update_layout(height=270)",
    "            st.plotly_chart(fig_g, use_container_width=True)",
    "",
    "# ==== TAB 4 CLUSTERING ========================================",
    "with tab4:",
    "    st.subheader(\"📦 Analisis Klaster K-Means\")",
    "    if \"kmeans\" not in mdl:",
    "        st.warning(\"Model clustering belum tersedia. Jalankan 04_clustering.ipynb.\")",
    "    else:",
    "        cl_feats = [c for c in mdl.get(\"cl_feats\", []) if c in df.columns]",
    "        df_cl = df[cl_feats + [\"Defect_Overall\", \"Material_Description\"]].dropna().copy()",
    "        if len(df_cl) > 0:",
    "            Xc  = mdl[\"sc_cl\"].transform(df_cl[cl_feats])",
    "            Xp  = mdl[\"pca\"].transform(Xc)",
    "            df_cl[\"Cluster\"] = mdl[\"kmeans\"].predict(Xc)",
    "            df_cl[\"Cluster_Label\"] = df_cl[\"Cluster\"].astype(str).map(mdl[\"cl_map\"]).fillna(df_cl[\"Cluster\"].astype(str))",
    "            df_cl[\"PCA1\"] = Xp[:,0]",
    "            df_cl[\"PCA2\"] = Xp[:,1]",
    "",
    "            cL, cR = st.columns(2)",
    "            with cL:",
    "                fig_pca = px.scatter(",
    "                    df_cl, x=\"PCA1\", y=\"PCA2\", color=\"Cluster_Label\",",
    "                    symbol=\"Defect_Overall\", hover_data=[\"Material_Description\"],",
    "                    title=\"PCA 2D – Klaster\",",
    "                    color_discrete_sequence=px.colors.qualitative.Set2",
    "                )",
    "                st.plotly_chart(fig_pca, use_container_width=True)",
    "            with cR:",
    "                cstat = (df_cl.groupby(\"Cluster_Label\")[\"Defect_Overall\"]",
    "                         .agg([\"sum\",\"count\"]).rename(columns={\"sum\":\"Defect\",\"count\":\"Total\"})",
    "                         .assign(Rate=lambda x: x[\"Defect\"]/x[\"Total\"]*100))",
    "                fig_cb = px.bar(",
    "                    cstat.reset_index(), x=\"Cluster_Label\", y=\"Rate\",",
    "                    color=\"Rate\", title=\"Defect Rate per Klaster (%)\",",
    "                    color_continuous_scale=\"RdYlGn_r\"",
    "                )",
    "                st.plotly_chart(fig_cb, use_container_width=True)",
    "",
    "            prof = df_cl.groupby(\"Cluster_Label\")[cl_feats].mean().round(3)",
    "            prof[\"Defect_Rate_%\"] = cstat[\"Rate\"].round(2)",
    "            prof[\"Jumlah_Batch\"]  = cstat[\"Total\"]",
    "            st.dataframe(prof.style.background_gradient(cmap=\"RdYlGn_r\", subset=[\"Defect_Rate_%\"]), use_container_width=True)",
    "",
    "# ==== TAB 5 INSIGHT ===========================================",
    "with tab5:",
    "    st.subheader(\"💡 Feature Importance & Rekomendasi\")",
    "    if \"rf\" in mdl and \"feats\" in mdl:",
    "        imp = pd.Series(mdl[\"rf\"].feature_importances_, index=mdl[\"feats\"]).sort_values(ascending=False)",
    "        fig_fi = px.bar(",
    "            imp.reset_index(), x=\"index\", y=0,",
    "            title=\"Feature Importance – Random Forest\",",
    "            labels={\"index\":\"Fitur\", \"0\":\"Importance\"},",
    "            color=imp.values, color_continuous_scale=\"Blues\"",
    "        )",
    "        fig_fi.update_layout(coloraxis_showscale=False)",
    "        st.plotly_chart(fig_fi, use_container_width=True)",
    "",
    "        st.markdown(\"### 🏆 Top 5 Faktor Penyebab Defect\")",
    "        for i, (feat, val) in enumerate(imp.head(5).items()):",
    "            pct = val / imp.sum() * 100",
    "            st.markdown(f\"**{i+1}. {feat.replace(chr(95), ' ')}** — {pct:.1f}%\")",
    "            st.progress(float(val / imp.max()))",
    "",
    "    st.divider()",
    "    st.markdown(\"### 🎯 Rekomendasi Operasional\")",
    "    for title, desc in [",
    "        (\"🔴 Monitoring Real-time\", \"Pantau batch Cetak_Pct_Teoritis < 90% otomatis.\"),",
    "        (\"🟡 Alert Sistem\", \"Kirim notifikasi QC jika probabilitas defect > 50%.\"),",
    "        (\"🟢 Root Cause\", \"Analisis mesin pada klaster High-Risk.\"),",
    "        (\"📊 Laporan Harian\", \"Generate laporan batch otomatis setiap hari.\"),",
    "        (\"🔧 Maintenance Preventif\", \"Jadwalkan perawatan mesin sebelum batch berisiko.\"),",
    "    ]:",
    "        st.info(f\"**{title}**: {desc}\")",
    "",
    "    num_corr = [c for c in [",
    "        \"GB_Yield_Total\", \"GK_Yield_Total\", \"Cetak_Pct_Teoritis\",",
    "        \"Kemas_Pct_Teoritis\", \"Total_Waste_Kg\", \"Defect_Overall\"",
    "    ] if c in df.columns]",
    "",
    "    if len(num_corr) > 2:",
    "        corr = df[num_corr].corr()",
    "        fig_corr = px.imshow(",
    "            corr, text_auto=\".2f\", color_continuous_scale=\"RdYlGn\",",
    "            title=\"Heatmap Korelasi\", zmin=-1, zmax=1",
    "        )",
    "        st.plotly_chart(fig_corr, use_container_width=True)",
    "",
    "st.divider()",
    "st.markdown(\"<center><small>💊 AI Pharma Monitoring | PJK-GM016 | Pijak × IBM SkillsBuild</small></center>\", unsafe_allow_html=True)",
]

with open("app.py", "w", encoding="utf-8") as fout:
    fout.write("\n".join(dashboard_lines))

print("✅ app.py berhasil dibuat!")
print(f"   Baris kode : {len(dashboard_lines)}")
print(f"   Ukuran     : {os.path.getsize('app.py'):,} bytes")
print("\\nUntuk menjalankan:")
print("   python -m pip install streamlit plotly")
print("   streamlit run app.py")

✅ app.py berhasil dibuat!
   Baris kode : 312
   Ukuran     : 14,874 bytes
\nUntuk menjalankan:
   python -m pip install streamlit plotly
   streamlit run app.py


## 🧪 3. Test Komponen (Tanpa Streamlit)

In [7]:
import joblib, pandas as pd, numpy as np

print("=" * 55)
print("  PENGUJIAN KOMPONEN DASHBOARD")
print("=" * 55)

# Test 1 – Load data
print("\n[1] Load Data...")
for f in ["rekap_produksi_with_predictions.csv","rekap_produksi_clean.csv"]:
    if os.path.exists(f):
        dft = pd.read_csv(f, low_memory=False)
        print(f"    ✅ {f}: {dft.shape}")
        break

# Test 2 – Load RF model
print("\n[2] Load Random Forest...")
try:
    rf_t  = joblib.load("models/random_forest_model.pkl")
    sc_t  = joblib.load("models/scaler.pkl")
    feats = open("models/feature_cols.txt").read().splitlines()
    print(f"    ✅ {rf_t.n_estimators} trees | {len(feats)} features")
except Exception as e:
    print(f"    ❌ {e}")

# Test 3 – Load K-Means
print("\n[3] Load K-Means...")
try:
    km = joblib.load("models/kmeans_model.pkl")
    print(f"    ✅ K={km.n_clusters}")
except Exception as e:
    print(f"    ❌ {e}")

# Test 4 – Simulasi prediksi
print("\n[4] Simulasi Prediksi...")
try:
    dummy = {
        "GB_Yield_Total":700, "GK_Yield_Total":650, "Rasio_GK_GB":0.93,
        "GB_Kadar_Air_Mean":4.5, "Cetak_Yield_Kg":780,
        "Cetak_Pct_Teoritis":0.975, "Kemas_Pct_Teoritis":0.97,
        "Total_Waste_Kg":0, "Cetak_Durasi_Hari":1,
        "Kemas_Durasi_Hari":1, "Bulan_Produksi":6
    }
    Xi  = np.array([[dummy.get(f,0) for f in feats]])
    Xsc = sc_t.transform(Xi)
    pred   = rf_t.predict(Xsc)[0]
    proba  = rf_t.predict_proba(Xsc)[0][1]
    result = "DEFECT" if pred==1 else "NORMAL"
    print(f"    ✅ Prediksi: {result} (prob defect={proba*100:.1f}%)")
except Exception as e:
    print(f"    ❌ {e}")

# Test 5 – KPI
print("\n[5] Kalkulasi KPI...")
try:
    total = len(dft)
    n_def = int(dft["Defect_Overall"].sum())
    dr    = n_def/total*100
    print(f"    ✅ Total: {total} | Defect: {n_def} | Rate: {dr:.2f}%")
except Exception as e:
    print(f"    ❌ {e}")

print("\n✅ Pengujian selesai.")

  PENGUJIAN KOMPONEN DASHBOARD

[1] Load Data...
    ✅ rekap_produksi_with_predictions.csv: (662, 71)

[2] Load Random Forest...
    ✅ 100 trees | 14 features

[3] Load K-Means...
    ✅ K=3

[4] Simulasi Prediksi...


c:\Users\ACER\anaconda3\envs\farmasi\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


    ✅ Prediksi: NORMAL (prob defect=0.9%)

[5] Kalkulasi KPI...
    ✅ Total: 662 | Defect: 48 | Rate: 7.25%

✅ Pengujian selesai.


## 📋 4. Checklist & Panduan Deploy

In [8]:
print("""
================================================================
  PANDUAN DEPLOYMENT – PJK-GM016
================================================================

CARA MENJALANKAN DASHBOARD:
  1. Pastikan semua file model & data sudah ada (cek cell 1)
  2. Install dependensi:
       pip install streamlit plotly scikit-learn pandas joblib
  3. Jalankan dashboard:
       streamlit run app.py
  4. Buka browser: http://localhost:8501

STRUKTUR FOLDER:
  project_pjkgm016/
  ├── app.py
  ├── rekap_produksi_clean.csv
  ├── rekap_produksi_with_predictions.csv
  ├── rekap_produksi_clustered.csv
  └── models/
      ├── random_forest_model.pkl
      ├── isolation_forest_model.pkl
      ├── scaler.pkl
      ├── feature_cols.txt
      ├── kmeans_model.pkl
      ├── scaler_clustering.pkl
      ├── pca_model.pkl
      ├── cluster_features.txt
      └── cluster_labels_map.json

CHECKLIST ALPHA TESTING (Internal Tim):
  ☐ Semua 5 tab terbuka tanpa error
  ☐ Filter sidebar (produk & bulan) berfungsi
  ☐ KPI cards menampilkan angka yang benar
  ☐ Grafik tren bisa di-zoom & hover
  ☐ Tombol Analisis Batch menghasilkan prediksi
  ☐ Gauge chart muncul dengan nilai yang benar
  ☐ PCA scatter clustering tampil
  ☐ Feature importance bar chart muncul

CHECKLIST BETA TESTING (Pengguna/Mentor):
  ☐ Tampilan responsif di berbagai layar
  ☐ Loading < 5 detik
  ☐ Instruksi UI mudah dipahami non-teknis
  ☐ Output prediksi dapat diinterpretasi
  ☐ Tidak ada crash saat filter berubah
  ☐ Feedback dikumpulkan via form/kuesioner
================================================================
  ✅ NOTEBOOK 05 SELESAI – Dashboard siap dijalankan!
================================================================
""")


  PANDUAN DEPLOYMENT – PJK-GM016

CARA MENJALANKAN DASHBOARD:
  1. Pastikan semua file model & data sudah ada (cek cell 1)
  2. Install dependensi:
       pip install streamlit plotly scikit-learn pandas joblib
  3. Jalankan dashboard:
       streamlit run app.py
  4. Buka browser: http://localhost:8501

STRUKTUR FOLDER:
  project_pjkgm016/
  ├── app.py
  ├── rekap_produksi_clean.csv
  ├── rekap_produksi_with_predictions.csv
  ├── rekap_produksi_clustered.csv
  └── models/
      ├── random_forest_model.pkl
      ├── isolation_forest_model.pkl
      ├── scaler.pkl
      ├── feature_cols.txt
      ├── kmeans_model.pkl
      ├── scaler_clustering.pkl
      ├── pca_model.pkl
      ├── cluster_features.txt
      └── cluster_labels_map.json

CHECKLIST ALPHA TESTING (Internal Tim):
  ☐ Semua 5 tab terbuka tanpa error
  ☐ Filter sidebar (produk & bulan) berfungsi
  ☐ KPI cards menampilkan angka yang benar
  ☐ Grafik tren bisa di-zoom & hover
  ☐ Tombol Analisis Batch menghasilkan prediksi
  ☐